# Task 1.1 — Basic Train/Test Split

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv('clean_dataset.csv')

# Regression target for this dataset: monthly_salary
X = df.drop(columns=['monthly_salary'])
y = df['monthly_salary']

# 80/20 split, random_state fixes the shuffle so results are reproducible
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Training set: {X_train.shape}')
print(f'Test set: {X_test.shape}')
# random_state=42 -> same rows go to train/test every run.
# Without it, each run gets a different split, so results can't be compared or reproduced.

Training set: (288, 18)
Test set: (72, 18)


# Task 1.2 — Three-Way Split (Train / Validation / Test)

In [2]:
# Train (60%) -> fit the model
# Validation (20%) -> tune hyperparameters, compare models
# Test (20%) -> FINAL evaluation only, touched once at the end

# Step 1: split off the test set first
X_temp, X_test2, y_temp, y_test2 = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Step 2: split the remainder into train/validation (0.25 * 0.8 = 0.2 of the original data)
X_train2, X_val, y_train2, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=42
)

print(f'Train: {X_train2.shape[0]} rows ({X_train2.shape[0] / len(X):.0%})')
print(f'Val:   {X_val.shape[0]} rows ({X_val.shape[0] / len(X):.0%})')
print(f'Test:  {X_test2.shape[0]} rows ({X_test2.shape[0] / len(X):.0%})')
# Test set is set aside here and not used again until Task 5 (final evaluation).

Train: 216 rows (60%)
Val:   72 rows (20%)
Test:  72 rows (20%)


# Task 1.3 — K-Fold Cross-Validation

In [3]:
from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import LinearRegression

# K-Fold: split data into 5 parts, train on 4, test on 1, repeat 5 times
kf = KFold(n_splits=5, shuffle=True, random_state=42)
model = LinearRegression()

scores = cross_val_score(model, X, y, cv=kf, scoring='r2')
print(f'R2 scores per fold: {scores}')
print(f'Mean R2: {scores.mean():.3f} (+/- {scores.std():.3f})')
# CV averages performance over 5 different splits instead of relying on a single
# lucky/unlucky split, so the estimate is less sensitive to which rows landed in test.
# A large std between folds also flags an unstable model.

R2 scores per fold: [0.66976206 0.44399173 0.20256625 0.02420519 0.65334767]
Mean R2: 0.399 (+/- 0.253)


# Task 1.4 — Stratified K-Fold (Classification)

In [4]:
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression

# Classification target: status_ABSENT (0/1 flag). Note 'is_absent' is the SAME
# information but was scaled/standardized in Week 3, so it can't be used as a
# classification label directly - status_ABSENT is the clean 0/1 version.
# Drop is_absent and status_PRESENT too, since they leak the target.
X_class = df.drop(columns=['is_absent', 'status_ABSENT', 'status_PRESENT'])
y_class = df['status_ABSENT']

# Check class balance before choosing a CV strategy
print('Class balance:')
print(y_class.value_counts(normalize=True))

# StratifiedKFold keeps the same class proportions in every fold.
# Use it instead of plain KFold whenever the target is categorical and/or imbalanced -
# a random KFold split could put almost all of the minority class in one fold.
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
clf = LogisticRegression(max_iter=1000)

scores_clf = cross_val_score(clf, X_class, y_class, cv=skf, scoring='accuracy')
print(f'Stratified CV accuracy: {scores_clf.mean():.3f} (+/- {scores_clf.std():.3f})')

Class balance:
status_ABSENT
0.0    0.902778
1.0    0.097222
Name: proportion, dtype: float64
Stratified CV accuracy: 1.000 (+/- 0.000)
